# 09. String Parsing and Simulation

"Implement a standard function" problems: `atoi`, adding numbers as strings, compressing, zigzag. They test **careful step-by-step logic** and edge cases, not clever algorithms.

**Learning order:** ATOI → ADD BINARY → MULTIPLY STRINGS → COMPARE VERSIONS → REVERSE WORDS → COMPRESS → ZIGZAG → INTERVIEW THINKING

## 1. String to Integer (atoi)

Rules: skip leading spaces, optional sign, read digits until a non-digit, **clamp** to the 32-bit range. Process in this exact order and say each rule out loud.

In [1]:
def my_atoi(s):
    i, n = 0, len(s)
    while i < n and s[i] == " ":
        i += 1
    sign = 1
    if i < n and s[i] in "+-":
        sign = -1 if s[i] == "-" else 1
        i += 1
    num = 0
    while i < n and s[i].isdigit():
        num = num * 10 + int(s[i])
        i += 1
    num *= sign
    return max(-2**31, min(2**31 - 1, num))

print(my_atoi("42"), my_atoi("   -42"), my_atoi("4193 with words"), my_atoi("words and 987"), my_atoi("-91283472332"))   # 42 -42 4193 0 -2147483648

42 -42 4193 0 -2147483648


## 2. Add Binary (and the carry pattern)

Work from the **right**, keep a `carry`, build digits in a list, reverse at the end. The same pattern solves *Add Strings*, *Add Two Numbers* (linked lists), *Plus One*.

In [2]:
def add_binary(a, b):
    i, j, carry, res = len(a) - 1, len(b) - 1, 0, []
    while i >= 0 or j >= 0 or carry:
        total = carry
        if i >= 0:
            total += int(a[i]); i -= 1
        if j >= 0:
            total += int(b[j]); j -= 1
        res.append(str(total % 2))
        carry = total // 2
    return "".join(reversed(res))

print(add_binary("11", "1"), add_binary("1010", "1011"))   # 100 10101

100 10101


## 3. Multiply Strings (school multiplication)

Digit `i` of `a` times digit `j` of `b` contributes to positions `i + j` and `i + j + 1` of the result. Accumulate in an array of length `len(a) + len(b)`, then handle carries.

In [3]:
def multiply(a, b):
    if a == "0" or b == "0":
        return "0"
    res = [0] * (len(a) + len(b))
    for i in range(len(a) - 1, -1, -1):
        for j in range(len(b) - 1, -1, -1):
            total = int(a[i]) * int(b[j]) + res[i + j + 1]
            res[i + j + 1] = total % 10
            res[i + j] += total // 10
    return "".join(map(str, res)).lstrip("0")

print(multiply("123", "456"), multiply("2", "3"), multiply("999", "999"))   # 56088 6 998001

56088 6 998001


## 4. Compare Version Numbers

Split by `.`, compare as **integers** (so `"01" == "1"`), padding the shorter list with zeros.

In [4]:
def compare_version(v1, v2):
    a, b = v1.split("."), v2.split(".")
    for i in range(max(len(a), len(b))):
        x = int(a[i]) if i < len(a) else 0
        y = int(b[i]) if i < len(b) else 0
        if x != y:
            return 1 if x > y else -1
    return 0

print(compare_version("1.01", "1.001"), compare_version("1.0", "1.0.0"), compare_version("0.1", "1.1"))   # 0 0 -1

0 0 -1


## 5. Reverse Words in a String

Python: `" ".join(s.split()[::-1])`. `split()` with no argument removes extra spaces. (Interview follow-up: do it **in place** on a character array: reverse everything, then reverse each word.)

In [5]:
def reverse_words(s):
    return " ".join(s.split()[::-1])

print(repr(reverse_words("  hello world  ")), reverse_words("a good   example"))   # 'world hello' example good a

'world hello' example good a


## 6. String Compression (in place on a list)

Replace runs of the same character by the character plus the run length (if > 1). Use two pointers: `read` scans runs, `write` writes the result. Return the new length.

In [6]:
def compress(chars):
    write = read = 0
    while read < len(chars):
        ch, start = chars[read], read
        while read < len(chars) and chars[read] == ch:
            read += 1
        chars[write] = ch
        write += 1
        if read - start > 1:
            for digit in str(read - start):
                chars[write] = digit
                write += 1
    return write

chars = ["a", "a", "b", "b", "c", "c", "c"]
k = compress(chars)
print(k, chars[:k])   # 6 ['a', '2', 'b', '2', 'c', '3']

6 ['a', '2', 'b', '2', 'c', '3']


## 7. Zigzag Conversion

Write the string in a zigzag across `numRows` rows and read it row by row. Keep the **current row** and a **direction** that flips at the first and last row.

In [7]:
def convert_zigzag(s, num_rows):
    if num_rows == 1 or num_rows >= len(s):
        return s
    rows, r, step = [""] * num_rows, 0, 1
    for ch in s:
        rows[r] += ch
        if r == 0:
            step = 1
        elif r == num_rows - 1:
            step = -1
        r += step
    return "".join(rows)

print(convert_zigzag("PAYPALISHIRING", 3))   # PAHNAPLSIIGYIR

PAHNAPLSIIGYIR


## 8. Interview thinking

**Say:** "I'll go step by step: ... First handle whitespace, then the sign, then digits, then clamp." Then **list the edge cases before coding**: empty string, only spaces, sign only, overflow, leading zeros, different lengths.

**Common mistakes:** forgetting the final carry, reversing digits wrongly, using `int()` on the whole input when the problem forbids it, not clamping, not handling `"0"`, mutating the input when not allowed.

## Quick Revision

- Parse in the exact order of the rules. Clamp at the end.
- Addition: right to left, `carry`, build a list, reverse.
- Multiplication: `res[i + j + 1]` accumulates, carry to `res[i + j]`.
- In-place compress: `read` / `write` pointers.
- List edge cases before coding.